In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# ACC feedback–cue encoding reactivation (All Subjects)

Parallel all-subject analysis for:

1. **ACC encoding contrast vectors** (pos vs neg, pos vs neutral) in MVPA-significant feedback windows and the cross-decoding window — saved per electrode for reload.
2. **Within-subject cosine similarity** between the two contrast vectors (dedicated cell).
3. **RPE-category encoding reactivation (PositiveRPE-style)**: window-mean feedback HFA; **one** 3-class Ridge `HFA ~ C(rpe_cat, Treatment('pos'))` per template; **label-shuffle surrogates** to z-score **per-electrode contrast weights**; cue reactivation = **cue HFA · w_z** (same method as `MemoryBandit-PositiveRPE_Reinstatement_SingleSubject.ipynb`). Training labels from **`learn_df`** (trial index = `trials` − 1). Split by **cue** RPE category for plots.
4. **HPC theta (power + phase) vs reactivation**: trial-wise ACC reactivation timecourses regressed against theta power (β, permutation z) and phase-binned MI (permutation z), **pos vs ref** contrast on cue RPE; separate loop over HPC electrodes (ACC and HPC channel counts can differ).
5. Trialwise reactivation saved per subject (NPZ + long-form); per-electrode **weights** CSV; HPC theta long CSV.
6. **combined_df merge**: per subject, `per_subject/<sid>/<sid>_combined_df_with_reactivation.csv` adds `react_pos_vs_neg_cross_cue`, `react_pos_vs_neutral_cross_cue`, raw variants, and `cue_window_*` (mean cue-time reactivation in the **cross-decoding significant cue window** parsed from `ACC_crossdecoding_stats.txt`, **cross_decoding** template only). Group file: `combined_df_with_reactivation.parquet` (CSV fallback).

**ACC HFA preprocessing** matches MVPA-Light ``test_script_feedback_decoding.m`` (70–201 Hz band-mean, ``movmean`` smoothing, ``|power|>10`` interpolation along time, then downsample). Tunables: ``HFA_SMOOTH_METHOD``, ``HFA_ARTIFACT_ABS_THRESH`` in the config cell.

**Debugging:** helpers in this notebook (`Subject` + theta extractors from `mv_regression_utils`). Run the **single-subject demo** cell before the parallel block.



In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import hashlib
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

import re
from patsy import dmatrix
from sklearn.linear_model import Ridge

from membandit.plotting_style import set_plot_style
from membandit.io import Subject
from membandit._impl.multivariate import extract_theta_power_from_cue_tfr, extract_theta_phase_from_cue_epochs, compute_time_resolved_theta_power_reactivation_beta, compute_time_resolved_theta_phase_reactivation_mi

warnings.filterwarnings('ignore')
set_plot_style()
sns.set_style('white')

# -----------------------------
# Config
# -----------------------------
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

ACC_ROI = 'ACC'
EVENT_ENCODING = 'feedback_start'
EVENT_PROJECTION = 'cue_on'

FREQ_RANGE = (70.0, 201.0)
DOWNSAMPLE_FACTOR = 5
WIN_MS = 200
# HFA preprocessing: match MVPA-Light MemoryBandit/test_script_feedback_decoding.m
HFA_SMOOTH_METHOD = 'movmean'
HFA_ARTIFACT_ABS_THRESH = 10.0
MIN_TRIALS = 10
SEED = 42

# MVPA / cross-decoding stats (parsed in helpers cell; fall back if missing)
MVPA_STATS_PATH = Path(f'{base_dir}/scratch/MemoryBandit/MVPA/decoding_feedback/ACC_rpe_category_hfa_200_stats.txt')
CROSSDECODING_STATS_PATH = Path(f'{base_dir}/scratch/MemoryBandit/MVPA/crossdecoding/ACC_crossdecoding_stats.txt')

RPE_ORDER = ['neg', 'neutral', 'pos']
CAT_MAP = {
    'negative': 'neg',
    'neg': 'neg',
    'neutral': 'neutral',
    'positive': 'pos',
    'pos': 'pos',
}
CATEGORY_COLORS = {
    'neutral': '#7f7f7f',
    'neg': '#d62728',
    'pos': '#2ca02c',
}

# PositiveRPE-style encoding: Ridge on 3-class design + weight z vs label shuffle
RIDGE_ALPHA = 1.0
N_SURROGATES = 250
SURR_SEED = 42
MIN_TRIALS_PER_CLASS = 6

# HPC theta vs reactivation (per-electrode HPC; ACC n_elec independent)
THETA_ROI = 'HPC'
THETA_BAND = (4.0, 8.0)
POWER_MIN_TRIALS = 6
PHASE_MIN_TRIALS = 8
POWER_N_PERM = 200
PHASE_N_PERM = 200
PHASE_N_BINS = 18
THETA_SEED = 42
RUN_HPC_THETA = True

# Single-subject demo (runs before parallel); None -> first eligible subject
DEMO_SUBJECT_ID = None
DEMO_PLOTS = True

analysis_output_root = Path('/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/analysis_outputs/hpc_theta_acc_reactivation_parallel_all_subjects')
figure_output_root = analysis_output_root / 'figures'
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)



In [ ]:
# -----------------------------
# Helpers (inlined for notebook debugging; Subject + theta I/O from mv_regression_utils)
# -----------------------------
import hashlib
import re
import warnings as _warnings
from typing import Any, Dict, List, Optional


def fit_encoding_model(X: np.ndarray, y: np.ndarray, alpha: float = 1.0):
    """Fit encoding model using Ridge regression (inlined from mv_regression_utils)."""
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    model = Ridge(alpha=alpha)
    model.fit(X, y)
    return model.coef_.astype(np.float32), np.float32(model.intercept_)


def _get_smoothed_event_signal(
    subject_obj: Subject,
    roi: str,
    event: str,
    freq_range,
    downsample_factor: int,
    win_ms: int,
):
    """Get smoothed/downsampled event signal from Subject cache for one ROI (inlined).

    Uses ``HFA_SMOOTH_METHOD`` / ``HFA_ARTIFACT_ABS_THRESH`` (config cell) to match
    MVPA-Light ``test_script_feedback_decoding.m`` via ``Subject._get_or_build_event_neural_data``.
    """
    neural = subject_obj._get_or_build_event_neural_data(
        event=event,
        roi=roi,
        freq_range=freq_range,
        downsample_factor=downsample_factor,
        win_ms=win_ms,
        smooth_method=HFA_SMOOTH_METHOD,
        artifact_abs_thresh=HFA_ARTIFACT_ABS_THRESH,
    )
    signal = np.asarray(neural['hfa'], dtype=np.float32)
    times = np.asarray(neural['times'], dtype=np.float32)

    electrode_labels = neural.get('electrode_labels')
    if electrode_labels is None:
        raise ValueError(
            f"Missing electrode_labels for {subject_obj.subject} {event}/{roi}; "
            "refusing synthetic label fallback."
        )
    electrode_labels = np.asarray(electrode_labels, dtype=str)
    return signal, times, electrode_labels


def _resolve_trial_var_for_event(df: pd.DataFrame, event: str) -> str:
    """Resolve trial index column for an event-specific dataframe (inlined)."""
    if event == 'cue_on':
        for candidate in ('trials_mem', 'trials_gamble', 'trials'):
            if candidate in df.columns:
                return candidate
    for candidate in ('trials', 'trials_gamble', 'trials_mem'):
        if candidate in df.columns:
            return candidate
    raise ValueError(f"Could not determine trial variable for event '{event}'")


def _build_lrt_trial_axis(
    subject_obj: Subject,
    event_projection: str,
    cue_n_trials: int,
) -> Dict[str, Any]:
    """Build LRT trial axis from Day1 combined_df (inlined from mv_regression_utils)."""
    memory_df = subject_obj.combined_df[
        (subject_obj.combined_df.participant == subject_obj.subject) &
        (subject_obj.combined_df.condition == 'Day1')
    ].copy()
    if len(memory_df) == 0:
        raise ValueError("no_day1_trials_for_lrt_axis")

    if 'memory_outcome' not in memory_df.columns:
        needed = ['hits', 'misses', 'false_alarms', 'correct_rejections']
        missing = [c for c in needed if c not in memory_df.columns]
        if len(missing) > 0:
            raise ValueError(f"missing_memory_columns_for_lrt_axis:{missing}")
        memory_df['memory_outcome'] = memory_df[needed].idxmax(axis=1)

    trial_col = _resolve_trial_var_for_event(memory_df, event_projection)
    memory_df = memory_df.dropna(subset=[trial_col, 'memory_outcome']).copy()
    if len(memory_df) == 0:
        raise ValueError("no_valid_trials_after_dropna_for_lrt_axis")

    trial_vals = pd.to_numeric(memory_df[trial_col], errors='coerce').to_numpy(dtype=np.float64)
    in_range = (
        np.isfinite(trial_vals) &
        ((trial_vals - 1) >= 0) &
        ((trial_vals - 1) < cue_n_trials)
    )
    if int(np.sum(in_range)) == 0:
        raise ValueError("no_in_range_trials_for_lrt_axis")

    trial_ix = (trial_vals[in_range].astype(np.int64) - 1)
    outcomes = memory_df.loc[in_range, 'memory_outcome'].astype(object).to_numpy()
    return {
        'trial_index_lrt': trial_ix.astype(np.int64, copy=False),
        'memory_outcomes_lrt': outcomes,
        'n_dropped_alignment': int((~in_range).sum()),
    }


def _parse_mvpa_decoding_feedback_clusters(stats_path: Path) -> list:
    """Parse 'Time range: a – b s' lines from ACC MVPA cluster stats."""
    text = stats_path.read_text(encoding='utf-8', errors='replace')
    pat = re.compile(r'Time range:\s*([\d.]+)\s*[–-]\s*([\d.]+)\s*s', re.IGNORECASE)
    pairs = [(float(a), float(b)) for a, b in pat.findall(text)]
    return pairs


def _parse_crossdecoding_feedback_range(stats_path: Path):
    text = stats_path.read_text(encoding='utf-8', errors='replace')
    m = re.search(
        r'Feedback time range:\s*([\d.]+)\s*[–-]\s*([\d.]+)\s*s',
        text,
        re.IGNORECASE,
    )
    if not m:
        return None
    return (float(m.group(1)), float(m.group(2)))


def _parse_crossdecoding_cue_range(stats_path: Path):
    text = stats_path.read_text(encoding='utf-8', errors='replace')
    m = re.search(
        r'Cue time range:\s*([\d.]+)\s*[–-]\s*([\d.]+)\s*s',
        text,
        re.IGNORECASE,
    )
    if not m:
        return None
    return (float(m.group(1)), float(m.group(2)))


MVPA_FB_RANGES_EXPECTED = [(0.28, 0.48), (0.75, 0.94)]
CROSS_FB_RANGE_EXPECTED = (0.81, 1.50)
CROSS_CUE_RANGE_EXPECTED = (0.30, 0.64)

if MVPA_STATS_PATH.exists():
    MVPA_DECODING_FB_RANGES = _parse_mvpa_decoding_feedback_clusters(MVPA_STATS_PATH)
else:
    MVPA_DECODING_FB_RANGES = list(MVPA_FB_RANGES_EXPECTED)
    _warnings.warn(f'MVPA stats not found: {MVPA_STATS_PATH}; using hard-coded windows.')

if CROSSDECODING_STATS_PATH.exists():
    _cross = _parse_crossdecoding_feedback_range(CROSSDECODING_STATS_PATH)
    CROSS_DECODING_FB_RANGE = _cross if _cross is not None else CROSS_FB_RANGE_EXPECTED
    _cross_cue = _parse_crossdecoding_cue_range(CROSSDECODING_STATS_PATH)
    CROSS_DECODING_CUE_RANGE = _cross_cue if _cross_cue is not None else CROSS_CUE_RANGE_EXPECTED
else:
    CROSS_DECODING_FB_RANGE = CROSS_FB_RANGE_EXPECTED
    CROSS_DECODING_CUE_RANGE = CROSS_CUE_RANGE_EXPECTED
    _warnings.warn(
        f'Cross-decoding stats not found: {CROSSDECODING_STATS_PATH}; using hard-coded feedback + cue windows.'
    )


def _ranges_close(a, b, tol=0.002):
    if len(a) != len(b):
        return False
    for (x0, x1), (y0, y1) in zip(a, b):
        if abs(x0 - y0) > tol or abs(x1 - y1) > tol:
            return False
    return True


if not _ranges_close(MVPA_DECODING_FB_RANGES, MVPA_FB_RANGES_EXPECTED):
    _warnings.warn(
        f'Parsed MVPA feedback clusters {MVPA_DECODING_FB_RANGES} differ from expected '
        f'{MVPA_FB_RANGES_EXPECTED}; using parsed values.'
    )
if (
    abs(CROSS_DECODING_FB_RANGE[0] - CROSS_FB_RANGE_EXPECTED[0]) > 0.002
    or abs(CROSS_DECODING_FB_RANGE[1] - CROSS_FB_RANGE_EXPECTED[1]) > 0.002
):
    _warnings.warn(
        f'Parsed cross-decoding feedback window {CROSS_DECODING_FB_RANGE} differs from '
        f'expected {CROSS_FB_RANGE_EXPECTED}; using parsed values.'
    )
if (
    abs(CROSS_DECODING_CUE_RANGE[0] - CROSS_CUE_RANGE_EXPECTED[0]) > 0.002
    or abs(CROSS_DECODING_CUE_RANGE[1] - CROSS_CUE_RANGE_EXPECTED[1]) > 0.002
):
    _warnings.warn(
        f'Parsed cross-decoding cue window {CROSS_DECODING_CUE_RANGE} differs from '
        f'expected {CROSS_CUE_RANGE_EXPECTED}; using parsed values.'
    )


def _feedback_time_mask(fb_times: np.ndarray, ranges: list) -> np.ndarray:
    t = np.asarray(fb_times, dtype=np.float64).reshape(-1)
    mask = np.zeros(t.shape[0], dtype=bool)
    for t0, t1 in ranges:
        mask |= (t >= float(t0)) & (t <= float(t1))
    return mask


def _mean_over_feedback_time(
    fb_hfa: np.ndarray,
    fb_times: np.ndarray,
    ranges: list,
) -> np.ndarray:
    """Average HFA over feedback time bins in union of ranges -> (n_trials, n_electrodes)."""
    mask = _feedback_time_mask(fb_times, ranges)
    if not np.any(mask):
        raise ValueError(f'No feedback time bins in ranges {ranges}')
    x = np.asarray(fb_hfa, dtype=np.float64)
    return np.nanmean(x[:, :, mask], axis=2)


def _average_react_in_window(react: np.ndarray, cue_times: np.ndarray, window: tuple) -> np.ndarray:
    """Mean cue-time reactivation within [t0, t1]; one value per trial (rows of react)."""
    r = np.asarray(react, dtype=np.float64)
    t = np.asarray(cue_times, dtype=np.float64).reshape(-1)
    if r.ndim != 2 or r.shape[1] != t.size:
        raise ValueError(f'react shape {r.shape} inconsistent with n_times={t.size}')
    t0, t1 = float(window[0]), float(window[1])
    mask = (t >= t0) & (t <= t1)
    if not np.any(mask):
        return np.full(r.shape[0], np.nan, dtype=np.float64)
    return np.nanmean(r[:, mask], axis=1)


def _build_combined_df_with_reactivation(
    subject_obj: Subject,
    trial_index_lrt: np.ndarray,
    per_trial: Dict[str, np.ndarray],
    cue_window: tuple,
) -> pd.DataFrame:
    """Merge LRT-aligned cue reactivation scalars into Day1 combined_df (on trial_index_lrt)."""
    sid = subject_obj.subject
    cdf = subject_obj.combined_df[
        (subject_obj.combined_df.participant == sid)
        & (subject_obj.combined_df.condition == 'Day1')
    ].copy()
    if len(cdf) == 0:
        return cdf
    trial_col = None
    for candidate in ('trials_mem', 'trials_gamble', 'trials'):
        if candidate in cdf.columns:
            trial_col = candidate
            break
    if trial_col is None:
        return cdf
    cdf['trial_index_lrt'] = pd.to_numeric(cdf[trial_col], errors='coerce').astype('Int64') - 1
    n = int(len(trial_index_lrt))
    add: Dict[str, Any] = {'trial_index_lrt': trial_index_lrt.astype(int)}
    for k, v in per_trial.items():
        arr = np.asarray(v, dtype=np.float64).reshape(-1)
        if arr.size < n:
            vv = np.full(n, np.nan, dtype=np.float64)
            vv[: arr.size] = arr
            add[k] = vv
        else:
            add[k] = arr[:n]
    lrt = pd.DataFrame(add)
    lrt['cue_window_start_s'] = float(cue_window[0])
    lrt['cue_window_end_s'] = float(cue_window[1])
    return cdf.merge(lrt, on='trial_index_lrt', how='left')


def _build_feedback_rpe_cat_from_learn(subject_obj: Subject, n_trials: int) -> np.ndarray:
    """Map learn_df RPE categories onto feedback trial axis 0..n_trials-1 (sparse NaN)."""
    out = np.full(int(n_trials), np.nan, dtype=object)
    ld = getattr(subject_obj, 'learn_df', None)
    if ld is None or len(ld) == 0:
        return out
    learn_df = ld.loc[ld.participant == subject_obj.subject].copy()
    if len(learn_df) == 0 or 'trials' not in learn_df.columns:
        return out
    learn_df['rpe_cat_norm'] = (
        learn_df['rpe_cat'].astype(str).str.strip().str.lower().map(CAT_MAP)
    )
    trial_ix = pd.to_numeric(learn_df['trials'], errors='coerce').to_numpy(dtype=np.float64) - 1.0
    valid = (
        np.isfinite(trial_ix)
        & (trial_ix >= 0)
        & (trial_ix < n_trials)
        & learn_df['rpe_cat_norm'].notna()
        & learn_df['rpe_cat_norm'].isin(RPE_ORDER)
    )
    learn_df = learn_df.loc[valid].copy()
    trial_ix = trial_ix[valid].astype(np.int64)
    for ti, cat in zip(trial_ix, learn_df['rpe_cat_norm'].to_numpy()):
        out[int(ti)] = cat
    return out


def _compute_per_electrode_contrast(
    fb_hfa: np.ndarray,
    fb_times: np.ndarray,
    rpe_cat: np.ndarray,
    ranges: list,
    cat_pos: str,
    cat_ref: str,
) -> np.ndarray:
    """Mean(pos trials) - mean(ref trials) of time-averaged feedback HFA per electrode."""
    trial_means = _mean_over_feedback_time(fb_hfa, fb_times, ranges)
    mpos = (rpe_cat == cat_pos)
    mref = (rpe_cat == cat_ref)
    if np.sum(mpos) == 0 or np.sum(mref) == 0:
        return np.full(trial_means.shape[1], np.nan, dtype=np.float64)
    a = np.nanmean(trial_means[mpos, :], axis=0)
    b = np.nanmean(trial_means[mref, :], axis=0)
    return (a - b).astype(np.float64, copy=False)


def _normalize_rpe_cat_series(s: pd.Series) -> pd.Series:
    mapping = {
        'negative': 'neg',
        'neg': 'neg',
        'neutral': 'neutral',
        'positive': 'pos',
        'pos': 'pos',
    }
    return s.astype(str).str.strip().str.lower().map(mapping)


def _build_cue_rpe_cat_on_lrt_axis(subject_obj: Subject, trial_index_lrt: np.ndarray) -> np.ndarray:
    df = subject_obj.combined_df[
        (subject_obj.combined_df.participant == subject_obj.subject) &
        (subject_obj.combined_df.condition == 'Day1')
    ].copy()
    if len(df) == 0:
        return np.full(len(trial_index_lrt), np.nan, dtype=object)

    if 'rpe_cat' not in df.columns or df['rpe_cat'].isna().all():
        if 'rpe' in df.columns:
            vals = pd.to_numeric(df['rpe'], errors='coerce')
            try:
                df['rpe_cat'] = pd.qcut(vals, q=3, labels=['neg', 'neutral', 'pos'], duplicates='drop')
            except Exception:
                df['rpe_cat'] = np.nan
        else:
            df['rpe_cat'] = np.nan

    df['rpe_cat'] = _normalize_rpe_cat_series(df['rpe_cat'])

    trial_col = None
    for candidate in ('trials_mem', 'trials_gamble', 'trials'):
        if candidate in df.columns:
            trial_col = candidate
            break
    if trial_col is None:
        return np.full(len(trial_index_lrt), np.nan, dtype=object)

    trial_vals = pd.to_numeric(df[trial_col], errors='coerce')
    keep = np.isfinite(trial_vals) & df['rpe_cat'].notna()
    df = df.loc[keep].copy()
    if len(df) == 0:
        return np.full(len(trial_index_lrt), np.nan, dtype=object)

    df['trial_ix'] = trial_vals.loc[df.index].astype(int) - 1
    trial_to_cat = (
        df.sort_values('trial_ix')
          .drop_duplicates(subset=['trial_ix'], keep='last')
          .set_index('trial_ix')['rpe_cat']
          .to_dict()
    )
    return np.asarray([trial_to_cat.get(int(ix), np.nan) for ix in np.asarray(trial_index_lrt, dtype=int)], dtype=object)


def _trial_meta_lrt_dataframe(
    subject_id: str,
    trial_index_lrt: np.ndarray,
    cue_rpe_cat_lrt: np.ndarray,
    subject_obj: Subject,
) -> pd.DataFrame:
    """One row per LRT-aligned cue trial for post-hoc joins."""
    n = len(trial_index_lrt)
    base = pd.DataFrame({
        'subject': subject_id,
        'lrt_row_idx': np.arange(n, dtype=int),
        'trial_index_lrt': trial_index_lrt.astype(int),
        'cue_rpe_cat': [str(x) if pd.notna(x) and str(x) != 'nan' else '' for x in cue_rpe_cat_lrt],
    })
    cdf = subject_obj.combined_df[
        (subject_obj.combined_df.participant == subject_obj.subject) &
        (subject_obj.combined_df.condition == 'Day1')
    ].copy()
    if len(cdf) == 0:
        return base
    trial_col = None
    for candidate in ('trials_mem', 'trials_gamble', 'trials'):
        if candidate in cdf.columns:
            trial_col = candidate
            break
    if trial_col is None:
        return base
    cdf = cdf.copy()
    cdf['trial_ix_merge'] = pd.to_numeric(cdf[trial_col], errors='coerce').astype('Int64') - 1
    cdf = cdf.dropna(subset=['trial_ix_merge'])
    cdf = cdf.sort_values('trial_ix_merge').drop_duplicates(subset=['trial_ix_merge'], keep='last')
    join_cols = [c for c in cdf.columns if c not in base.columns and c != trial_col]
    skip = {'participant', 'condition', 'trial_ix_merge', 'subject'}
    join_cols = [c for c in join_cols if c not in skip][:40]
    merged = base.merge(
        cdf[['trial_ix_merge'] + join_cols].rename(columns={'trial_ix_merge': 'trial_index_lrt'}),
        on='trial_index_lrt',
        how='left',
    )
    return merged


def _subject_surr_seed(subject_id: str) -> int:
    h = int(hashlib.md5(str(subject_id).encode('utf-8')).hexdigest()[:8], 16)
    return int(SURR_SEED + (h % (2**31)))


def _run_window_encoding_pospref(
    fb_pattern: np.ndarray,
    fb_rpe_cat: np.ndarray,
    cue_hfa: np.ndarray,
    subject_id: str,
    alpha: float = 1.0,
    n_surrogates: int = 250,
):
    """3-class Ridge (Treatment pos); z-score w_pos_vs_* vs label-shuffle; reactivation = cue·w_z."""
    fb_pattern = np.asarray(fb_pattern, dtype=np.float64)
    cue_hfa = np.asarray(cue_hfa, dtype=np.float32)
    rc = np.asarray(fb_rpe_cat, dtype=object)
    row_ok = np.isfinite(fb_pattern).all(axis=1) & pd.Series(rc).isin(list(RPE_ORDER)).to_numpy()
    if int(np.sum(row_ok)) < int(MIN_TRIALS):
        return None
    Y_all = fb_pattern[row_ok]
    rc_ok = rc[row_ok]
    df_design = pd.DataFrame({'rpe_cat': pd.Categorical(rc_ok, categories=RPE_ORDER, ordered=False)})
    try:
        X_all = np.asarray(
            dmatrix("C(rpe_cat, Treatment('pos'))", df_design, return_type='dataframe'),
            dtype=np.float64,
        )
    except Exception:
        return None
    row_ok2 = np.isfinite(X_all).all(axis=1) & np.isfinite(Y_all).all(axis=1)
    X_fit = X_all[row_ok2]
    Y_fit = Y_all[row_ok2]
    rc_fit = rc_ok[row_ok2]
    counts = {c: int(np.sum(rc_fit == c)) for c in RPE_ORDER}
    if any(counts[c] < int(MIN_TRIALS_PER_CLASS) for c in RPE_ORDER):
        return None
    n_fit = int(X_fit.shape[0])
    if n_fit < int(MIN_TRIALS):
        return None

    model = Ridge(alpha=float(alpha), fit_intercept=False)
    model.fit(X_fit, Y_fit)
    coef = np.asarray(model.coef_, dtype=np.float64)
    w_pos_vs_neg = -coef[:, 1].astype(np.float64, copy=False)
    w_pos_vs_neutral = -coef[:, 2].astype(np.float64, copy=False)

    rng = np.random.default_rng(_subject_surr_seed(subject_id))
    n_elec = int(w_pos_vs_neg.size)
    surr_neg = np.zeros((int(n_surrogates), n_elec), dtype=np.float64)
    surr_neu = np.zeros((int(n_surrogates), n_elec), dtype=np.float64)
    for s in range(int(n_surrogates)):
        perm = rng.permutation(n_fit)
        rpe_perm = rc_fit[perm]
        df_p = pd.DataFrame({'rpe_cat': pd.Categorical(rpe_perm, categories=RPE_ORDER, ordered=False)})
        X_p = np.asarray(
            dmatrix("C(rpe_cat, Treatment('pos'))", df_p, return_type='dataframe'),
            dtype=np.float64,
        )
        m_s = Ridge(alpha=float(alpha), fit_intercept=False)
        m_s.fit(X_p, Y_fit)
        c_s = np.asarray(m_s.coef_, dtype=np.float64)
        surr_neg[s, :] = -c_s[:, 1]
        surr_neu[s, :] = -c_s[:, 2]

    std_neg = np.maximum(np.std(surr_neg, axis=0, ddof=1), 1e-8)
    std_neu = np.maximum(np.std(surr_neu, axis=0, ddof=1), 1e-8)
    w_z_neg = (w_pos_vs_neg - np.mean(surr_neg, axis=0)) / std_neg
    w_z_neutral = (w_pos_vs_neutral - np.mean(surr_neu, axis=0)) / std_neu

    react_neg = np.asarray(
        np.tensordot(cue_hfa, w_z_neg.astype(np.float32), axes=(1, 0)),
        dtype=np.float32,
    )
    react_neutral = np.asarray(
        np.tensordot(cue_hfa, w_z_neutral.astype(np.float32), axes=(1, 0)),
        dtype=np.float32,
    )
    react_raw_neg = np.asarray(
        np.tensordot(cue_hfa, w_pos_vs_neg.astype(np.float32), axes=(1, 0)),
        dtype=np.float32,
    )
    react_raw_neutral = np.asarray(
        np.tensordot(cue_hfa, w_pos_vs_neutral.astype(np.float32), axes=(1, 0)),
        dtype=np.float32,
    )

    return {
        'class_counts': counts,
        'n_fit': n_fit,
        'w_raw': {'pos_vs_neg': w_pos_vs_neg, 'pos_vs_neutral': w_pos_vs_neutral},
        'w_z': {'pos_vs_neg': w_z_neg, 'pos_vs_neutral': w_z_neutral},
        'react': {'pos_vs_neg': react_neg, 'pos_vs_neutral': react_neutral},
        'react_raw': {'pos_vs_neg': react_raw_neg, 'pos_vs_neutral': react_raw_neutral},
    }


def _prepare_acc_lrt_for_subject(subject_id: str):
    """Load ACC HFA, align feedback/cue on LRT axis. Returns dict with ok=False on failure."""
    try:
        subject_obj = Subject(subject_id, base_dir, load_dir, min_trials=MIN_TRIALS, verbose=False)

        fb_hfa, fb_times, fb_labels = _get_smoothed_event_signal(
            subject_obj=subject_obj,
            roi=ACC_ROI,
            event=EVENT_ENCODING,
            freq_range=FREQ_RANGE,
            downsample_factor=DOWNSAMPLE_FACTOR,
            win_ms=WIN_MS,
        )
        cue_hfa, cue_times, cue_labels = _get_smoothed_event_signal(
            subject_obj=subject_obj,
            roi=ACC_ROI,
            event=EVENT_PROJECTION,
            freq_range=FREQ_RANGE,
            downsample_factor=DOWNSAMPLE_FACTOR,
            win_ms=WIN_MS,
        )

        common_acc = [lab for lab in fb_labels.astype(str) if lab in set(cue_labels.astype(str))]
        if len(common_acc) == 0:
            return {'ok': False, 'error': 'No common ACC electrodes'}

        fb_ix = np.asarray([np.where(fb_labels.astype(str) == lab)[0][0] for lab in common_acc], dtype=int)
        cue_ix = np.asarray([np.where(cue_labels.astype(str) == lab)[0][0] for lab in common_acc], dtype=int)

        fb_hfa = np.asarray(fb_hfa[:, fb_ix, :], dtype=np.float32)
        cue_hfa = np.asarray(cue_hfa[:, cue_ix, :], dtype=np.float32)

        fb_rpe_cat = _build_feedback_rpe_cat_from_learn(subject_obj, fb_hfa.shape[0])

        lrt_axis = _build_lrt_trial_axis(
            subject_obj=subject_obj,
            event_projection=EVENT_PROJECTION,
            cue_n_trials=cue_hfa.shape[0],
        )
        trial_index_lrt = np.asarray(lrt_axis['trial_index_lrt'], dtype=int)

        in_range = (
            (trial_index_lrt >= 0)
            & (trial_index_lrt < cue_hfa.shape[0])
            & (trial_index_lrt < fb_hfa.shape[0])
        )
        trial_index_lrt = trial_index_lrt[in_range]
        if trial_index_lrt.size == 0:
            return {'ok': False, 'error': 'No valid LRT trial indices'}

        fb_hfa_lrt = fb_hfa[trial_index_lrt, :, :]
        cue_hfa_lrt = cue_hfa[trial_index_lrt, :, :]
        cue_rpe_cat_lrt = _build_cue_rpe_cat_on_lrt_axis(subject_obj, trial_index_lrt)

        return {
            'ok': True,
            'subject_obj': subject_obj,
            'fb_hfa': fb_hfa,
            'fb_times': fb_times,
            'fb_labels': fb_labels,
            'cue_hfa_lrt': cue_hfa_lrt,
            'cue_times': cue_times,
            'cue_labels': cue_labels,
            'trial_index_lrt': trial_index_lrt,
            'cue_rpe_cat_lrt': cue_rpe_cat_lrt,
            'fb_rpe_cat': fb_rpe_cat,
            'common_acc': common_acc,
        }
    except Exception as exc:
        return {'ok': False, 'error': repr(exc)}


def _compute_acc_rpecat_encoding_bundle(
    subject_id: str,
    subject_obj: Subject,
    fb_hfa: np.ndarray,
    fb_times: np.ndarray,
    cue_hfa_lrt: np.ndarray,
    cue_times: np.ndarray,
    trial_index_lrt: np.ndarray,
    cue_rpe_cat_lrt: np.ndarray,
    fb_rpe_cat: np.ndarray,
    common_acc: list,
    include_debug: bool = False,
):
    """Contrast vectors + PositiveRPE-style encoding reactivation + weights + trialwise outputs."""
    contrast_rows = []
    weight_rows = []
    agg_rows = []
    npz_payload = {'cue_times': np.asarray(cue_times, dtype=np.float32)}
    trialwise_dfs = []
    debug = {} if include_debug else None

    TEMPLATES = (
        ('mvpa_decoding', list(MVPA_DECODING_FB_RANGES)),
        ('cross_decoding', [CROSS_DECODING_FB_RANGE]),
    )
    CONTRASTS = (
        ('pos_vs_neg', ('pos', 'neg')),
        ('pos_vs_neutral', ('pos', 'neutral')),
    )

    for template_name, ranges in TEMPLATES:
        for contrast_name, (cp, cr) in CONTRASTS:
            vec = _compute_per_electrode_contrast(
                fb_hfa, fb_times, fb_rpe_cat, list(ranges), cp, cr,
            )
            for ei, lab in enumerate(common_acc):
                val = float(vec[ei]) if ei < len(vec) else np.nan
                if np.isfinite(val):
                    contrast_rows.append({
                        'subject': subject_id,
                        'electrode': str(lab),
                        'template': template_name,
                        'contrast': contrast_name,
                        'value': val,
                    })

    for template_name, ranges in TEMPLATES:
        fb_window = _mean_over_feedback_time(fb_hfa, fb_times, list(ranges))
        if include_debug:
            debug.setdefault('_fb_window', {})[template_name] = fb_window.astype(np.float32)

        rep = _run_window_encoding_pospref(
            fb_window,
            fb_rpe_cat,
            cue_hfa_lrt,
            subject_id,
            alpha=RIDGE_ALPHA,
            n_surrogates=N_SURROGATES,
        )
        if rep is None:
            continue

        n_fit = int(rep['n_fit'])
        class_counts = rep['class_counts']

        for contrast_name, (cat_pos, cat_ref) in CONTRASTS:
            react = rep['react'][contrast_name]
            react_raw = rep['react_raw'][contrast_name]
            w_raw = rep['w_raw'][contrast_name]
            w_z = rep['w_z'][contrast_name]
            n_pos_tr = int(class_counts.get('pos', 0))
            n_ref_tr = int(class_counts.get(cat_ref, 0))
            key_meta = f'{contrast_name}__{template_name}'
            npz_payload[f'{key_meta}__n_fit'] = np.array([n_fit], dtype=np.int32)
            npz_payload[f'{key_meta}__n_pos_fb'] = np.array([n_pos_tr], dtype=np.int32)
            npz_payload[f'{key_meta}__n_ref_fb'] = np.array([n_ref_tr], dtype=np.int32)

            zk = f'{contrast_name}__{template_name}__z'
            rk = f'{contrast_name}__{template_name}__real'
            npz_payload[zk] = react
            npz_payload[rk] = react_raw

            for ei, lab in enumerate(common_acc):
                weight_rows.append({
                    'subject': subject_id,
                    'electrode': str(lab),
                    'template': template_name,
                    'contrast': contrast_name,
                    'w_raw': float(w_raw[ei]) if ei < len(w_raw) else np.nan,
                    'w_z': float(w_z[ei]) if ei < len(w_z) else np.nan,
                })

            if include_debug:
                mpos = (fb_rpe_cat == cat_pos)
                mref = (fb_rpe_cat == cat_ref)
                raw_cv = np.nanmean(fb_window[mpos, :], axis=0) - np.nanmean(fb_window[mref, :], axis=0)
                dnode = debug.setdefault(template_name, {}).setdefault(contrast_name, {})
                dnode.update(
                    react_z=react,
                    react_raw=react_raw,
                    w_raw=w_raw.astype(np.float32),
                    w_z=w_z.astype(np.float32),
                    raw_contrast_per_elec=raw_cv.astype(np.float32),
                    n_fit=n_fit,
                    class_counts=class_counts,
                )

            for cat in RPE_ORDER:
                cat_mask = (cue_rpe_cat_lrt == cat)
                if np.sum(cat_mask) == 0:
                    continue
                ymean = np.nanmean(react[cat_mask, :], axis=0)
                for t_idx, t_s in enumerate(cue_times):
                    val = ymean[t_idx]
                    if np.isfinite(val):
                        agg_rows.append({
                            'subject': subject_id,
                            'contrast': contrast_name,
                            'template': template_name,
                            'cue_rpe_cat': cat,
                            'time_idx': int(t_idx),
                            'time_s': float(t_s),
                            'reactivation': float(val),
                            'n_cue_trials_cat': int(np.sum(cat_mask)),
                            'n_pos_train': n_pos_tr,
                            'n_ref_train': n_ref_tr,
                            'n_fit_feedback': n_fit,
                        })

            n_tr, n_t = react.shape
            ti = np.repeat(np.arange(n_tr, dtype=int), n_t)
            tt = np.tile(np.arange(n_t, dtype=int), n_tr)
            crc = np.asarray(cue_rpe_cat_lrt, dtype=object)
            crc_s = np.array([str(crc[i]) if pd.notna(crc[i]) else '' for i in range(len(crc))], dtype=object)
            trialwise_dfs.append(pd.DataFrame({
                'subject': subject_id,
                'contrast': contrast_name,
                'template': template_name,
                'lrt_row_idx': ti,
                'trial_index_lrt': trial_index_lrt[ti],
                'cue_rpe_cat': crc_s[ti],
                'time_idx': tt,
                'time_s': cue_times[tt].astype(np.float64),
                'reactivation': react[ti, tt].astype(np.float64),
                'reactivation_raw': react_raw[ti, tt].astype(np.float64),
            }))

    trial_meta_df = _trial_meta_lrt_dataframe(subject_id, trial_index_lrt, cue_rpe_cat_lrt, subject_obj)
    trialwise_df = pd.concat(trialwise_dfs, ignore_index=True) if trialwise_dfs else pd.DataFrame()

    cue_win = (float(CROSS_DECODING_CUE_RANGE[0]), float(CROSS_DECODING_CUE_RANGE[1]))
    ct = np.asarray(cue_times, dtype=float)
    _react_keys = (
        ('pos_vs_neg__cross_decoding__z', 'react_pos_vs_neg_cross_cue'),
        ('pos_vs_neutral__cross_decoding__z', 'react_pos_vs_neutral_cross_cue'),
        ('pos_vs_neg__cross_decoding__real', 'react_raw_pos_vs_neg_cross_cue'),
        ('pos_vs_neutral__cross_decoding__real', 'react_raw_pos_vs_neutral_cross_cue'),
    )
    per_trial: Dict[str, np.ndarray] = {}
    if all(sk in npz_payload for sk, _ in _react_keys):
        for src_k, dst_k in _react_keys:
            per_trial[dst_k] = _average_react_in_window(npz_payload[src_k], ct, cue_win)
    else:
        n_lrt = int(len(trial_index_lrt))
        for _, dst_k in _react_keys:
            per_trial[dst_k] = np.full(n_lrt, np.nan, dtype=np.float64)

    combined_df_with_reactivation = _build_combined_df_with_reactivation(
        subject_obj, trial_index_lrt, per_trial, cue_win,
    )

    return {
        'contrast_rows': contrast_rows,
        'weight_rows': weight_rows,
        'agg_rows': agg_rows,
        'trial_meta_df': trial_meta_df,
        'npz_payload': npz_payload,
        'trialwise_df': trialwise_df,
        'debug': debug,
        'combined_df_with_reactivation': combined_df_with_reactivation,
        'cue_window_used': cue_win,
    }


def _load_hpc_theta_aligned(
    subject_obj: Subject,
    cue_times: np.ndarray,
    trial_index_lrt: np.ndarray,
) -> Optional[Dict[str, Any]]:
    """HPC theta power (TFR) + phase (epochs) on cue grid; returns None dict on failure."""
    try:
        power_res = extract_theta_power_from_cue_tfr(
            subject_obj=subject_obj,
            roi=THETA_ROI,
            event=EVENT_PROJECTION,
            freq_band=THETA_BAND,
            target_times=cue_times,
            trial_index=trial_index_lrt,
        )
        phase_res = extract_theta_phase_from_cue_epochs(
            subject_obj=subject_obj,
            roi=THETA_ROI,
            event=EVENT_PROJECTION,
            freq_band=THETA_BAND,
            target_times=cue_times,
            trial_index=trial_index_lrt,
        )
    except Exception as exc:
        return {'ok': False, 'error': repr(exc)}

    power_labels = np.asarray(power_res['electrode_labels'], dtype=str)
    phase_labels = np.asarray(phase_res['electrode_labels'], dtype=str)
    common_hpc = [lab for lab in power_labels.astype(str) if lab in set(phase_labels.astype(str))]
    if len(common_hpc) == 0:
        return {'ok': False, 'error': 'no_common_hpc_electrodes'}

    p_ix = np.asarray([np.where(power_labels.astype(str) == lab)[0][0] for lab in common_hpc], dtype=int)
    ph_ix = np.asarray([np.where(phase_labels.astype(str) == lab)[0][0] for lab in common_hpc], dtype=int)
    theta_power = np.asarray(power_res['power'][p_ix, :, :], dtype=np.float64)
    theta_phase = np.asarray(phase_res['phase'][ph_ix, :, :], dtype=np.float64)
    target_times = np.asarray(power_res['target_times'], dtype=np.float32)
    return {
        'ok': True,
        'theta_power': theta_power,
        'theta_phase': theta_phase,
        'common_hpc': common_hpc,
        'target_times': target_times,
    }


def _hpc_theta_rows_for_reactivation(
    subject_id: str,
    template_name: str,
    contrast_name: str,
    cat_pos: str,
    cat_ref: str,
    react: np.ndarray,
    theta_pack: Dict[str, Any],
    cue_rpe_cat_lrt: np.ndarray,
) -> List[Dict[str, Any]]:
    """react (n_trials, n_cue_t); returns long rows per HPC channel x time."""
    if not theta_pack.get('ok'):
        return []
    theta_power = theta_pack['theta_power']
    theta_phase = theta_pack['theta_phase']
    common_hpc = theta_pack['common_hpc']

    react = np.asarray(react, dtype=np.float64)
    if react.ndim != 2:
        return []
    react_t = react.T
    n_t_r, n_tr_r = react_t.shape
    n_t_h, n_tr_h = theta_power.shape[1], theta_power.shape[2]
    n_t = min(n_t_r, n_t_h)
    n_tr = min(n_tr_r, n_tr_h, len(cue_rpe_cat_lrt))
    react_t = react_t[:n_t, :n_tr]
    theta_power = theta_power[:, :n_t, :n_tr]
    theta_phase = theta_phase[:, :n_t, :n_tr]
    cue_rc = np.asarray(cue_rpe_cat_lrt[:n_tr], dtype=object)
    power_times = np.asarray(theta_pack['target_times'], dtype=float)[:n_t]

    mask_pos = (cue_rc == cat_pos)
    mask_ref = (cue_rc == cat_ref)
    if int(np.sum(mask_pos)) < POWER_MIN_TRIALS or int(np.sum(mask_ref)) < POWER_MIN_TRIALS:
        return []

    rows: List[Dict[str, Any]] = []
    seed_base = THETA_SEED + abs(hash((subject_id, template_name, contrast_name))) % (2**20)

    for ch_idx, ch_name in enumerate(common_hpc):
        ch_pow = theta_power[ch_idx, :, :]
        ch_phase = theta_phase[ch_idx, :, :]
        pow_pos = compute_time_resolved_theta_power_reactivation_beta(
            theta_power=ch_pow,
            reactivation=react_t,
            trial_mask=mask_pos,
            min_trials=POWER_MIN_TRIALS,
            n_permutations=POWER_N_PERM,
            seed=seed_base + ch_idx,
        )
        pow_ref = compute_time_resolved_theta_power_reactivation_beta(
            theta_power=ch_pow,
            reactivation=react_t,
            trial_mask=mask_ref,
            min_trials=POWER_MIN_TRIALS,
            n_permutations=POWER_N_PERM,
            seed=seed_base + ch_idx + 10000,
        )
        ph_pos = compute_time_resolved_theta_phase_reactivation_mi(
            theta_phase=ch_phase,
            reactivation=react_t,
            trial_mask=mask_pos,
            min_trials=PHASE_MIN_TRIALS,
            n_bins=PHASE_N_BINS,
            n_permutations=PHASE_N_PERM,
            seed=seed_base + ch_idx + 20000,
        )
        ph_ref = compute_time_resolved_theta_phase_reactivation_mi(
            theta_phase=ch_phase,
            reactivation=react_t,
            trial_mask=mask_ref,
            min_trials=PHASE_MIN_TRIALS,
            n_bins=PHASE_N_BINS,
            n_permutations=PHASE_N_PERM,
            seed=seed_base + ch_idx + 30000,
        )
        z_beta_diff = np.asarray(pow_pos['z_beta'], dtype=float) - np.asarray(pow_ref['z_beta'], dtype=float)
        z_mi_diff = np.asarray(ph_pos['z_mi'], dtype=float) - np.asarray(ph_ref['z_mi'], dtype=float)

        for t_idx, t_s in enumerate(power_times):
            b = z_beta_diff[t_idx] if t_idx < len(z_beta_diff) else np.nan
            m = z_mi_diff[t_idx] if t_idx < len(z_mi_diff) else np.nan
            if not (np.isfinite(b) or np.isfinite(m)):
                continue
            rows.append({
                'subject': subject_id,
                'template': template_name,
                'contrast': contrast_name,
                'theta_channel': str(ch_name),
                'time_idx': int(t_idx),
                'time_s': float(t_s),
                'z_beta_diff': float(b) if np.isfinite(b) else np.nan,
                'z_mi_diff': float(m) if np.isfinite(m) else np.nan,
                'n_pos_trials': int(np.sum(mask_pos)),
                'n_ref_trials': int(np.sum(mask_ref)),
            })
    return rows



## All-subject parallel execution

This section runs per-subject **ACC encoding contrast vectors**, **RPE-category encoding reactivation** (PositiveRPE-style: 3-class Ridge on window-mean feedback HFA, label-shuffle **z-scored weights**, cue reactivation = cue HFA · w_z; training labels from `learn_df`), **HPC theta** (power β and phase MI vs trial-wise reactivation, pos−ref on cue RPE), saves trialwise outputs and weights CSVs, then aggregates for group plots.

Run the **single-subject demo** cell first for diagnostics.


In [ ]:
# -----------------------------
# Subject list + parallel config
# -----------------------------
import os
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# Reuse the same eligibility logic used in other MemoryBandit notebooks.
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()
subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

# joblib.Parallel over subjects; -1 -> all cores. Set to a small int if I/O bound.
N_JOBS = -1
JOBLIB_BACKEND = 'loky'

parallel_output_root = analysis_output_root / 'all_subject_parallel'
parallel_figure_root = parallel_output_root / 'figures'
per_subject_output_root = parallel_output_root / 'per_subject'
parallel_output_root.mkdir(parents=True, exist_ok=True)
parallel_figure_root.mkdir(parents=True, exist_ok=True)
per_subject_output_root.mkdir(parents=True, exist_ok=True)

print('Eligible subjects:', len(subjects))
print('First 10 subjects:', subjects[:10])
print('Parallel workers (joblib N_JOBS):', N_JOBS, '| backend:', JOBLIB_BACKEND)
print('Parallel output root:', parallel_output_root)



In [ ]:
# -----------------------------
# Single-subject demo (debug) — run before parallel block
# -----------------------------
_demo_sid = DEMO_SUBJECT_ID if DEMO_SUBJECT_ID else (subjects[0] if len(subjects) else None)

if _demo_sid is None:
    print('Demo skipped: no eligible subjects.')
elif not DEMO_PLOTS:
    print('Demo skipped: DEMO_PLOTS is False.')
else:
    print('Demo subject:', _demo_sid)
    _prep = _prepare_acc_lrt_for_subject(_demo_sid)
    if not _prep['ok']:
        print('Demo prep failed:', _prep['error'])
    else:
        _bundle = _compute_acc_rpecat_encoding_bundle(
            _demo_sid,
            _prep['subject_obj'],
            _prep['fb_hfa'],
            _prep['fb_times'],
            _prep['cue_hfa_lrt'],
            _prep['cue_times'],
            _prep['trial_index_lrt'],
            _prep['cue_rpe_cat_lrt'],
            _prep['fb_rpe_cat'],
            _prep['common_acc'],
            include_debug=True,
        )
        _fb_rc = np.asarray(_prep['fb_rpe_cat'], dtype=object)
        _cue_rc = np.asarray(_prep['cue_rpe_cat_lrt'], dtype=object)
        _dbg = _bundle['debug']

        # --- 1) counts ---
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, title, arr in zip(
            axes,
            ['Feedback trials (learn-mapped rpe_cat)', 'LRT cue trials'],
            [_fb_rc, _cue_rc],
        ):
            cats, cnts = np.unique(np.array([str(x) if pd.notna(x) else 'nan' for x in arr]), return_counts=True)
            ax.bar(cats, cnts)
            ax.set_title(title)
            ax.tick_params(axis='x', rotation=45)
        fig.suptitle(f'{_demo_sid} | trial counts')
        fig.tight_layout()
        plt.show()

        # --- 2) feedback window pattern heatmaps (sorted by fb rpe_cat) ---
        for _tmpl, _ranges in (('mvpa_decoding', list(MVPA_DECODING_FB_RANGES)), ('cross_decoding', [CROSS_DECODING_FB_RANGE])):
            _pat = _mean_over_feedback_time(_prep['fb_hfa'], _prep['fb_times'], list(_ranges))
            _order = np.argsort(_fb_rc.astype(str))
            fig, ax = plt.subplots(figsize=(10, 6))
            im = ax.imshow(_pat[_order, :], aspect='auto', interpolation='nearest')
            ax.set_title(f'{_demo_sid} | window-mean feedback HFA | {_tmpl}\nrows sorted by feedback rpe_cat (learn)')
            ax.set_xlabel('electrode (ACC)')
            ax.set_ylabel('trial (sorted)')
            fig.colorbar(im, ax=ax, fraction=0.02)
            fig.tight_layout()
            plt.show()

        # --- 3) per (template, contrast): raw contrast vs w_raw / w_z ---
        for _tmpl in ('mvpa_decoding', 'cross_decoding'):
            if _tmpl not in _dbg:
                continue
            for _ctr in ('pos_vs_neg', 'pos_vs_neutral'):
                if _ctr not in _dbg[_tmpl]:
                    continue
                d = _dbg[_tmpl][_ctr]
                w_raw = d['w_raw']
                w_z = d['w_z']
                raw = d['raw_contrast_per_elec']
                fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
                x = np.arange(len(w_z))
                axes[0].bar(x, raw, alpha=0.35, label='mean(pos)-mean(ref)')
                axes[0].plot(x, w_raw, color='k', lw=1.5, label='Ridge w_raw')
                axes[0].legend()
                axes[0].set_ylabel('HFA / weight')
                axes[0].set_title(f'{_demo_sid} | {_tmpl} | {_ctr} | raw weights')
                axes[1].bar(x, w_z, alpha=0.8, color='steelblue')
                axes[1].axhline(0.0, color='gray', lw=0.8)
                axes[1].set_ylabel('w_z (vs label-shuffle)')
                axes[1].set_title('Surrogate z-scored contrast weights (per electrode)')
                fig.tight_layout()
                plt.show()

                rz = d['react_z']
                fig, ax = plt.subplots(figsize=(10, 5))
                _order2 = np.argsort(_cue_rc.astype(str))
                im = ax.imshow(rz[_order2, :], aspect='auto', interpolation='nearest')
                ax.set_title(f'{_demo_sid} | cue·w_z reactivation (sorted by cue rpe_cat)\n{_tmpl} | {_ctr}')
                ax.set_xlabel('cue time bin')
                ax.set_ylabel('LRT trial (sorted)')
                fig.colorbar(im, ax=ax, fraction=0.02)
                fig.tight_layout()
                plt.show()

        # --- HPC theta demo (first <=6 channels) ---
        _th = _load_hpc_theta_aligned(_prep['subject_obj'], _prep['cue_times'], _prep['trial_index_lrt'])
        if RUN_HPC_THETA and _th and _th.get('ok'):
            _hlabels = _th['common_hpc'][:6]
            for _tmpl in ('mvpa_decoding', 'cross_decoding'):
                if _tmpl not in _dbg:
                    continue
                for _ctr in ('pos_vs_neg', 'pos_vs_neutral'):
                    if _ctr not in _dbg[_tmpl]:
                        continue
                    react = _dbg[_tmpl][_ctr]['react_z']
                    cp, cr = ('pos', 'neg') if _ctr == 'pos_vs_neg' else ('pos', 'neutral')
                    _rows = _hpc_theta_rows_for_reactivation(
                        _demo_sid, _tmpl, _ctr, cp, cr, react, _th, _prep['cue_rpe_cat_lrt'],
                    )
                    if not _rows:
                        continue
                    _tdf = pd.DataFrame(_rows)
                    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
                    for ch in _hlabels:
                        sub = _tdf[_tdf['theta_channel'].astype(str) == str(ch)]
                        if len(sub) == 0:
                            continue
                        sub = sub.sort_values('time_s')
                        axes[0].plot(sub['time_s'], sub['z_beta_diff'], lw=1.2, label=str(ch))
                        axes[1].plot(sub['time_s'], sub['z_mi_diff'], lw=1.2, label=str(ch))
                    axes[0].axhline(0, color='k', lw=0.6)
                    axes[1].axhline(0, color='k', lw=0.6)
                    axes[0].set_ylabel('z_beta_diff (pos−ref)')
                    axes[1].set_ylabel('z_mi_diff (pos−ref)')
                    axes[0].set_title(f'HPC theta vs reactivation | {_tmpl} | {_ctr} (≤6 ch)')
                    axes[1].set_xlabel('Cue time (s)')
                    axes[0].legend(fontsize=7, ncol=2)
                    fig.tight_layout()
                    plt.show()
        elif RUN_HPC_THETA:
            print('HPC theta demo skip:', _th.get('error', _th) if isinstance(_th, dict) else _th)

        # --- mini group-style plot for this subject ---
        _adf = pd.DataFrame(_bundle['agg_rows'])
        if len(_adf):
            fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex='col')
            for i, _ctr in enumerate(['pos_vs_neg', 'pos_vs_neutral']):
                for j, _tmpl in enumerate(['mvpa_decoding', 'cross_decoding']):
                    ax = axes[i, j]
                    sub = _adf[(_adf['contrast'] == _ctr) & (_adf['template'] == _tmpl)]
                    for cat in RPE_ORDER:
                        s2 = sub[sub['cue_rpe_cat'] == cat]
                        if len(s2) == 0:
                            continue
                        g = s2.groupby('time_s', as_index=False)['reactivation'].agg(m='mean', sem='sem')
                        ax.plot(g['time_s'], g['m'], color=CATEGORY_COLORS.get(cat), lw=2, label=cat)
                        ax.fill_between(g['time_s'], g['m'] - g['sem'], g['m'] + g['sem'], color=CATEGORY_COLORS.get(cat), alpha=0.15)
                    ax.axhline(0.0, color='k', ls='--', lw=0.6)
                    ax.set_title(f'{_ctr}\n{_tmpl}')
                    ax.set_ylabel('cue·w_z')
            axes[1, 0].set_xlabel('Cue time (s)')
            axes[1, 1].set_xlabel('Cue time (s)')
            handles, labels = [], []
            for ax in axes.ravel():
                h, lab = ax.get_legend_handles_labels()
                if h:
                    handles, labels = h, lab
                    break
            if handles:
                fig.legend(handles, labels, title='Cue RPE', loc='upper right', bbox_to_anchor=(0.99, 0.98))
            fig.suptitle(f'{_demo_sid} | encoding reactivation (demo)', y=1.02)
            fig.tight_layout()
            plt.show()

        # --- save demo NPZ ---
        _demo_npz = {}
        for k, v in _bundle['npz_payload'].items():
            _demo_npz[k] = v
        for _tmpl, d1 in _dbg.items():
            if _tmpl.startswith('_'):
                continue
            for _ctr, d in d1.items():
                prefix = f'debug__{_tmpl}__{_ctr}__'
                for kk in ('w_raw', 'w_z', 'raw_contrast_per_elec', 'react_z', 'react_raw'):
                    if kk in d:
                        _demo_npz[prefix + kk] = np.asarray(d[kk])
        if RUN_HPC_THETA and _th and _th.get('ok'):
            _demo_npz['demo_theta_power'] = _th['theta_power'].astype(np.float32)
            _demo_npz['demo_theta_phase'] = _th['theta_phase'].astype(np.float32)
            _demo_npz['demo_hpc_labels'] = np.array(_th['common_hpc'], dtype=object)
        _out_demo = per_subject_output_root / f'{_demo_sid}_demo_rpecat_encoding_debug.npz'
        np.savez_compressed(_out_demo, **_demo_npz)
        print('Saved demo NPZ:', _out_demo)

        _cw = _bundle.get('cue_window_used')
        _cdfwr = _bundle.get('combined_df_with_reactivation')
        print('Cross-decoding cue window used for scalar reactivation (cross_decoding template):', _cw)
        if isinstance(_cdfwr, pd.DataFrame) and len(_cdfwr) and 'react_pos_vs_neg_cross_cue' in _cdfwr.columns:
            display(_cdfwr.head(10))
            _merge_plot = pd.DataFrame({
                'trial_index_lrt': np.asarray(_prep['trial_index_lrt'], dtype=int),
                'cue_rpe_cat': [str(x) if pd.notna(x) else '' for x in _prep['cue_rpe_cat_lrt']],
            })
            _cols_m = ['trial_index_lrt', 'react_pos_vs_neg_cross_cue', 'react_pos_vs_neutral_cross_cue']
            _mer = _merge_plot.merge(
                _cdfwr[[c for c in _cols_m if c in _cdfwr.columns]],
                on='trial_index_lrt',
                how='left',
            )
            fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
            for ax, col, title in zip(
                axes,
                ['react_pos_vs_neg_cross_cue', 'react_pos_vs_neutral_cross_cue'],
                ['pos vs neg (cue window mean)', 'pos vs neutral (cue window mean)'],
            ):
                if col not in _mer.columns:
                    continue
                for cat in RPE_ORDER:
                    m = _mer['cue_rpe_cat'].astype(str) == str(cat)
                    if not m.any():
                        continue
                    ax.scatter(
                        _mer.loc[m, 'trial_index_lrt'],
                        _mer.loc[m, col],
                        c=CATEGORY_COLORS.get(cat, 'k'),
                        label=cat,
                        alpha=0.65,
                        s=24,
                    )
                ax.axhline(0, color='gray', lw=0.7)
                ax.set_xlabel('trial_index_lrt')
                ax.set_ylabel('reactivation (cue window)')
                ax.set_title(f'{_demo_sid} | {title}')
            axes[0].legend(title='cue RPE', fontsize=8)
            fig.suptitle(
                'Cross-decoding template: mean cue HFA dot z-weight in significant cross-decoding cue window',
                y=1.02,
            )
            fig.tight_layout()
            plt.show()
        elif isinstance(_cdfwr, pd.DataFrame):
            print('combined_df_with_reactivation: missing reactivation columns (cross_decoding encoding failed?)')

        del _prep, _bundle, _dbg, _demo_npz


In [ ]:
# -----------------------------
# Per-subject worker + parallel execution (ACC encoding contrasts + RPE-category encoding reactivation + HPC theta)
# -----------------------------
def _run_subject_pipeline(subject_id: str) -> dict:
    prep = _prepare_acc_lrt_for_subject(subject_id)
    if not prep.get('ok'):
        return {
            'subject': subject_id,
            'status': 'error',
            'error': prep.get('error', 'unknown'),
            'contrast_rows': [],
            'weight_rows': [],
            'rpecat_agg_rows': [],
            'rpecat_trial_meta': None,
            'rpecat_npz_payload': None,
            'rpecat_trialwise_df': None,
            'hpc_theta_rows': [],
            'n_acc_common': np.nan,
            'n_trials_lrt': np.nan,
            'hpc_theta_error': '',
            'combined_df_with_reactivation': None,
            'n_combined_with_reactivation_rows': 0,
            'cue_window_used': None,
        }
    try:
        bundle = _compute_acc_rpecat_encoding_bundle(
            subject_id,
            prep['subject_obj'],
            prep['fb_hfa'],
            prep['fb_times'],
            prep['cue_hfa_lrt'],
            prep['cue_times'],
            prep['trial_index_lrt'],
            prep['cue_rpe_cat_lrt'],
            prep['fb_rpe_cat'],
            prep['common_acc'],
            include_debug=False,
        )
        cdwr = bundle.get('combined_df_with_reactivation')
        if isinstance(cdwr, pd.DataFrame) and len(cdwr) > 0:
            out_wr = per_subject_output_root / subject_id
            out_wr.mkdir(parents=True, exist_ok=True)
            cdwr.to_csv(out_wr / f'{subject_id}_combined_df_with_reactivation.csv', index=False)
        else:
            cdwr = None

        hpc_theta_rows = []
        hpc_theta_error = ''
        if RUN_HPC_THETA:
            th = _load_hpc_theta_aligned(
                prep['subject_obj'], prep['cue_times'], prep['trial_index_lrt'],
            )
            if not th.get('ok'):
                hpc_theta_error = str(th.get('error', 'unknown'))
            else:
                pay = bundle['npz_payload']
                CONTRAST_SPEC = (
                    ('pos_vs_neg', 'pos', 'neg'),
                    ('pos_vs_neutral', 'pos', 'neutral'),
                )
                for template_name in ('mvpa_decoding', 'cross_decoding'):
                    for cname, cp, cr in CONTRAST_SPEC:
                        zk = f'{cname}__{template_name}__z'
                        if zk not in pay:
                            continue
                        react = pay[zk]
                        hpc_theta_rows.extend(
                            _hpc_theta_rows_for_reactivation(
                                subject_id, template_name, cname, cp, cr, react, th, prep['cue_rpe_cat_lrt'],
                            )
                        )
        return {
            'subject': subject_id,
            'status': 'ok',
            'error': '',
            'n_acc_common': int(len(prep['common_acc'])),
            'n_trials_lrt': int(len(prep['trial_index_lrt'])),
            'contrast_rows': bundle['contrast_rows'],
            'weight_rows': bundle['weight_rows'],
            'rpecat_agg_rows': bundle['agg_rows'],
            'rpecat_trial_meta': bundle['trial_meta_df'],
            'rpecat_npz_payload': bundle['npz_payload'],
            'rpecat_trialwise_df': bundle['trialwise_df'],
            'hpc_theta_rows': hpc_theta_rows,
            'hpc_theta_error': hpc_theta_error,
            'combined_df_with_reactivation': cdwr,
            'n_combined_with_reactivation_rows': int(len(cdwr)) if isinstance(cdwr, pd.DataFrame) else 0,
            'cue_window_used': bundle.get('cue_window_used'),
        }
    except Exception as exc:
        return {
            'subject': subject_id,
            'status': 'error',
            'error': repr(exc),
            'contrast_rows': [],
            'weight_rows': [],
            'rpecat_agg_rows': [],
            'rpecat_trial_meta': None,
            'rpecat_npz_payload': None,
            'rpecat_trialwise_df': None,
            'hpc_theta_rows': [],
            'n_acc_common': np.nan,
            'n_trials_lrt': np.nan,
            'hpc_theta_error': '',
            'combined_df_with_reactivation': None,
            'n_combined_with_reactivation_rows': 0,
            'cue_window_used': None,
        }


results = Parallel(n_jobs=N_JOBS, backend=JOBLIB_BACKEND, verbose=5)(
    delayed(_run_subject_pipeline)(sid) for sid in tqdm(subjects, desc='subjects')
)
results = sorted(results, key=lambda x: x.get('subject', ''))

audit_rows = []
contrast_frames = []
weight_frames = []
rpecat_agg_frames = []
rpecat_tw_frames = []
hpc_theta_frames = []
combined_wr_frames = []
for res in results:
    twdf = res.get('rpecat_trialwise_df')
    n_tw = int(len(twdf)) if isinstance(twdf, pd.DataFrame) else 0
    htr = res.get('hpc_theta_rows') or []
    audit_rows.append({
        'subject': res.get('subject'),
        'status': res.get('status'),
        'error': res.get('error', ''),
        'n_acc_common': res.get('n_acc_common', np.nan),
        'n_trials_lrt': res.get('n_trials_lrt', np.nan),
        'n_contrast_rows': len(res.get('contrast_rows', [])),
        'n_weight_rows': len(res.get('weight_rows', [])),
        'n_rpecat_agg_rows': len(res.get('rpecat_agg_rows', [])),
        'n_rpecat_trialwise_rows': n_tw,
        'n_hpc_theta_rows': len(htr),
        'hpc_theta_error': res.get('hpc_theta_error', ''),
        'n_combined_with_reactivation_rows': int(res.get('n_combined_with_reactivation_rows', 0)),
        'cue_window_used': res.get('cue_window_used'),
    })

    if res.get('status') != 'ok':
        continue

    sid = res['subject']

    cdwr = res.get('combined_df_with_reactivation')
    if isinstance(cdwr, pd.DataFrame) and len(cdwr) > 0:
        _cdf = cdwr.copy()
        if 'subject' not in _cdf.columns:
            _cdf.insert(0, 'subject', sid)
        combined_wr_frames.append(_cdf)

    if len(res.get('contrast_rows', [])) > 0:
        subj_ctr_df = pd.DataFrame(res['contrast_rows'])
        contrast_frames.append(subj_ctr_df)
        subj_ctr_df.to_csv(
            per_subject_output_root / f'{sid}_acc_contrast_vectors_long.csv',
            index=False,
        )

    if len(res.get('weight_rows', [])) > 0:
        subj_w_df = pd.DataFrame(res['weight_rows'])
        weight_frames.append(subj_w_df)
        subj_w_df.to_csv(
            per_subject_output_root / f'{sid}_acc_rpecat_encoding_weights.csv',
            index=False,
        )

    pay = res.get('rpecat_npz_payload')
    if isinstance(pay, dict) and len(pay) > 0:
        np.savez_compressed(
            per_subject_output_root / f'{sid}_acc_rpecat_encoding_reinstatement_trialwise.npz',
            **pay,
        )

    meta = res.get('rpecat_trial_meta')
    if isinstance(meta, pd.DataFrame) and len(meta) > 0:
        meta.to_csv(
            per_subject_output_root / f'{sid}_acc_rpecat_encoding_reinstatement_trial_meta.csv',
            index=False,
        )

    if len(res.get('rpecat_agg_rows', [])) > 0:
        subj_ra = pd.DataFrame(res['rpecat_agg_rows'])
        rpecat_agg_frames.append(subj_ra)
        subj_ra.to_csv(
            per_subject_output_root / f'{sid}_acc_rpecat_encoding_reinstatement_long.csv',
            index=False,
        )

    if isinstance(twdf, pd.DataFrame) and len(twdf) > 0:
        rpecat_tw_frames.append(twdf)
        try:
            twdf.to_parquet(
                per_subject_output_root / f'{sid}_acc_rpecat_encoding_reinstatement_trialwise.parquet',
                index=False,
            )
        except Exception:
            twdf.to_csv(
                per_subject_output_root / f'{sid}_acc_rpecat_encoding_reinstatement_trialwise_long.csv',
                index=False,
            )

    if len(htr) > 0:
        hdf = pd.DataFrame(htr)
        hpc_theta_frames.append(hdf)
        hdf.to_csv(
            per_subject_output_root / f'{sid}_hpc_theta_reactivation_long.csv',
            index=False,
        )

audit_df = pd.DataFrame(audit_rows)
contrast_all_subjects_df = pd.concat(contrast_frames, ignore_index=True) if contrast_frames else pd.DataFrame()
weights_all_subjects_df = pd.concat(weight_frames, ignore_index=True) if weight_frames else pd.DataFrame()
rpecat_agg_all_subjects_df = pd.concat(rpecat_agg_frames, ignore_index=True) if rpecat_agg_frames else pd.DataFrame()
rpecat_trialwise_all_subjects_df = pd.concat(rpecat_tw_frames, ignore_index=True) if rpecat_tw_frames else pd.DataFrame()
hpc_theta_all_subjects_df = pd.concat(hpc_theta_frames, ignore_index=True) if hpc_theta_frames else pd.DataFrame()
combined_df_reactivation_all_subjects_df = (
    pd.concat(combined_wr_frames, ignore_index=True) if combined_wr_frames else pd.DataFrame()
)

if len(contrast_all_subjects_df):
    contrast_all_subjects_df.to_csv(parallel_output_root / 'acc_contrast_vectors_long.csv', index=False)
if len(weights_all_subjects_df):
    weights_all_subjects_df.to_csv(parallel_output_root / 'acc_rpecat_encoding_weights.csv', index=False)
if len(rpecat_agg_all_subjects_df):
    rpecat_agg_all_subjects_df.to_csv(
        parallel_output_root / 'acc_rpecat_encoding_reinstatement_long.csv',
        index=False,
    )
if len(rpecat_trialwise_all_subjects_df):
    try:
        rpecat_trialwise_all_subjects_df.to_parquet(
            parallel_output_root / 'acc_rpecat_encoding_reinstatement_trialwise.parquet',
            index=False,
        )
    except Exception as _e:
        print('Group trialwise parquet unavailable:', _e)
        for sid, g in rpecat_trialwise_all_subjects_df.groupby('subject'):
            g.to_csv(
                parallel_output_root / f'{sid}_acc_rpecat_encoding_reinstatement_trialwise_long.csv',
                index=False,
            )

if len(hpc_theta_all_subjects_df):
    hpc_theta_all_subjects_df.to_csv(
        parallel_output_root / 'hpc_theta_reactivation_long.csv',
        index=False,
    )
    try:
        hpc_theta_all_subjects_df.to_parquet(
            parallel_output_root / 'hpc_theta_reactivation_long.parquet',
            index=False,
        )
    except Exception as _e:
        print('Group HPC theta parquet unavailable:', _e)
elif RUN_HPC_THETA:
    print('No HPC theta rows aggregated (check per-subject errors in audit_df).')

if len(combined_df_reactivation_all_subjects_df):
    try:
        combined_df_reactivation_all_subjects_df.to_parquet(
            parallel_output_root / 'combined_df_with_reactivation.parquet',
            index=False,
        )
    except Exception as _e:
        print('Group combined_df_with_reactivation parquet unavailable:', _e)
        combined_df_reactivation_all_subjects_df.to_csv(
            parallel_output_root / 'combined_df_with_reactivation.csv',
            index=False,
        )

print('Requested subjects:', len(subjects))
print('Successful subjects:', int((audit_df['status'] == 'ok').sum()))
print('Failed subjects:', int((audit_df['status'] != 'ok').sum()))

display(audit_df.sort_values(['status', 'subject']).reset_index(drop=True))


In [ ]:
# -----------------------------
# Save audit + group-level summary (no early/late cosine traces)
# -----------------------------
(audit_df.sort_values('subject')).to_csv(parallel_output_root / 'all_subjects_parallel_audit.csv', index=False)

failed_df = audit_df[audit_df['status'] != 'ok'].copy()
if len(failed_df):
    print('Failed subject details:')
    display(failed_df[['subject', 'error']].sort_values('subject').reset_index(drop=True))
else:
    print('No failed subjects.')

per_subject_figure_root = per_subject_output_root / 'figures'
per_subject_figure_root.mkdir(parents=True, exist_ok=True)

print('Per-subject outputs under:', per_subject_output_root)
print('Group rpecat long CSV rows:', len(rpecat_agg_all_subjects_df))
print('Group rpecat trialwise rows:', len(rpecat_trialwise_all_subjects_df))
print('Group encoding weights rows:', len(weights_all_subjects_df))
print('Group HPC theta rows:', len(hpc_theta_all_subjects_df))
print('Group combined_df + reactivation rows:', len(combined_df_reactivation_all_subjects_df))
if len(combined_df_reactivation_all_subjects_df):
    display(combined_df_reactivation_all_subjects_df.head(4))



In [ ]:
# -----------------------------
# ACC encoding contrasts: cosine similarity pos_vs_neg vs pos_vs_neutral (per subject)
# -----------------------------
from scipy.spatial.distance import cosine as _scipy_cosine_dist
from scipy import stats as scipy_stats


def _vector_cosine_similarity(v1, v2):
    v1 = np.asarray(v1, dtype=float)
    v2 = np.asarray(v2, dtype=float)
    m = np.isfinite(v1) & np.isfinite(v2)
    if np.sum(m) < 2:
        return np.nan
    a, b = v1[m], v2[m]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return np.nan
    return float(1.0 - _scipy_cosine_dist(a, b))


if len(contrast_all_subjects_df) == 0:
    print('No contrast vectors; run pipeline cell first.')
else:
    cos_rows = []
    for sid in sorted(contrast_all_subjects_df['subject'].astype(str).unique()):
        for template in ('mvpa_decoding', 'cross_decoding'):
            sub = contrast_all_subjects_df[
                (contrast_all_subjects_df['subject'] == sid)
                & (contrast_all_subjects_df['template'] == template)
            ]
            pv = sub[sub['contrast'] == 'pos_vs_neg'].set_index('electrode')['value']
            pn = sub[sub['contrast'] == 'pos_vs_neutral'].set_index('electrode')['value']
            elecs = sorted(set(pv.index.astype(str)) & set(pn.index.astype(str)))
            if not elecs:
                continue
            cosim = _vector_cosine_similarity(
                pv.loc[elecs].astype(float).values,
                pn.loc[elecs].astype(float).values,
            )
            cos_rows.append({'subject': sid, 'template': template, 'cosine_similarity': cosim})

    cos_sim_df = pd.DataFrame(cos_rows)
    cos_sim_df.to_csv(parallel_output_root / 'acc_contrast_cosine_similarity.csv', index=False)
    print('Saved:', parallel_output_root / 'acc_contrast_cosine_similarity.csv')

    for template in ('mvpa_decoding', 'cross_decoding'):
        vals = cos_sim_df[cos_sim_df['template'] == template]['cosine_similarity'].dropna().to_numpy(dtype=float)
        print(f'\n{template}: n={len(vals)}, median={np.nanmedian(vals):.4f}')
        if len(vals) >= 6:
            try:
                w = scipy_stats.wilcoxon(vals, alternative='two-sided')
                print(f'  Wilcoxon signed-rank vs 0: statistic={w.statistic:.4g}, p={w.pvalue:.4g}')
            except Exception as exc:
                print(f'  Wilcoxon failed: {exc}')

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.boxplot(data=cos_sim_df, x='template', y='cosine_similarity', ax=ax, width=0.35)
    sns.stripplot(
        data=cos_sim_df,
        x='template',
        y='cosine_similarity',
        ax=ax,
        color='0.25',
        alpha=0.75,
        jitter=0.2,
        dodge=False,
    )
    ax.axhline(0.0, color='k', ls='--', lw=1)
    ax.set_ylabel('Cosine similarity')
    ax.set_xlabel('Feedback template')
    ax.set_title('Within-subject similarity of ACC contrast vectors\n(pos_vs_neg vs pos_vs_neutral)')
    fig.tight_layout()
    out_cos = parallel_figure_root / 'all_subjects_acc_contrast_cosine_distribution.pdf'
    fig.savefig(out_cos, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', out_cos)

    ctr_fig_root = parallel_figure_root / 'acc_contrast_per_subject'
    ctr_fig_root.mkdir(parents=True, exist_ok=True)
    for sid in sorted(contrast_all_subjects_df['subject'].astype(str).unique()):
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for ax, template in zip(axes, ('mvpa_decoding', 'cross_decoding')):
            sub = contrast_all_subjects_df[
                (contrast_all_subjects_df['subject'] == sid)
                & (contrast_all_subjects_df['template'] == template)
            ]
            if len(sub) == 0:
                ax.set_title(f'{template}\n(no data)')
                continue
            for cn, ls in (('pos_vs_neg', '-'), ('pos_vs_neutral', '--')):
                s2 = sub[sub['contrast'] == cn].sort_values('electrode').reset_index(drop=True)
                if len(s2) == 0:
                    continue
                x = np.arange(len(s2))
                ax.plot(x, s2['value'].values, ls=ls, lw=1.2, label=cn)
            s_ref = sub[sub['contrast'] == 'pos_vs_neg'].sort_values('electrode').reset_index(drop=True)
            if len(s_ref) == 0:
                s_ref = sub[sub['contrast'] == 'pos_vs_neutral'].sort_values('electrode').reset_index(drop=True)
            if len(s_ref) == 0:
                ax.set_title(f'{template}\n(no contrast rows)')
                continue
            step = max(1, len(s_ref) // 12)
            tick_idx = np.arange(0, len(s_ref), step)
            ax.set_xticks(tick_idx)
            ax.set_xticklabels(s_ref['electrode'].values[tick_idx], rotation=75, ha='right', fontsize=7)
            ax.axhline(0.0, color='k', lw=0.5)
            ax.set_title(template)
            ax.legend(fontsize=8)
            ax.set_ylabel('Contrast (mean HFA)')
            ax.set_xlabel('Electrode (sorted label)')

        fig.suptitle(f'{sid} | ACC encoding contrasts by electrode', y=1.02)
        fig.tight_layout()
        fig.savefig(ctr_fig_root / f'{sid}_acc_contrast_vectors.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
    print('Saved per-subject contrast figures to:', ctr_fig_root)

    for template in ('mvpa_decoding', 'cross_decoding'):
        sub = contrast_all_subjects_df[contrast_all_subjects_df['template'] == template].copy()
        if len(sub) == 0:
            continue
        wide = sub.pivot_table(index=['subject', 'electrode'], columns='contrast', values='value').reset_index()
        wide = wide.dropna(subset=['pos_vs_neg', 'pos_vs_neutral'])
        if len(wide) == 0:
            continue
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        ax.scatter(wide['pos_vs_neg'], wide['pos_vs_neutral'], s=14, alpha=0.35, edgecolors='none')
        lims = np.nanpercentile(
            np.c_[wide['pos_vs_neg'].values, wide['pos_vs_neutral'].values],
            [1, 99],
        ).flatten()
        lo, hi = float(np.min(lims)), float(np.max(lims))
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.7)
        ax.set_xlabel('pos_vs_neg')
        ax.set_ylabel('pos_vs_neutral')
        ax.set_title(f'All (subject, electrode) | {template}')
        ax.set_aspect('equal', adjustable='box')
        fig.tight_layout()
        fig.savefig(parallel_figure_root / f'all_subjects_acc_contrast_scatter_{template}.pdf', dpi=200, bbox_inches='tight')
        plt.close(fig)
        print('Saved:', parallel_figure_root / f'all_subjects_acc_contrast_scatter_{template}.pdf')


In [ ]:
# -----------------------------
# Group + per-subject: RPE-category encoding reactivation (cue HFA · z-weight)
# -----------------------------
if len(rpecat_agg_all_subjects_df) == 0:
    print('No rpecat encoding rows; run pipeline cell first.')
else:
    contrasts = ['pos_vs_neg', 'pos_vs_neutral']
    templates = ['mvpa_decoding', 'cross_decoding']
    fig, axes = plt.subplots(len(contrasts), len(templates), figsize=(10, 7), sharex='col')
    for i, ctr in enumerate(contrasts):
        for j, tmpl in enumerate(templates):
            ax = axes[i, j]
            sub_all = rpecat_agg_all_subjects_df[
                (rpecat_agg_all_subjects_df['contrast'] == ctr)
                & (rpecat_agg_all_subjects_df['template'] == tmpl)
            ]
            if len(sub_all) == 0:
                ax.set_title(f'{ctr} | {tmpl}\n(no data)')
                continue
            for cat in RPE_ORDER:
                parts = []
                for sid, g in sub_all[sub_all['cue_rpe_cat'] == cat].groupby('subject'):
                    gg = g.groupby('time_s', as_index=False)['reactivation'].mean()
                    if len(gg):
                        parts.append(gg.rename(columns={'reactivation': 'm'}))
                if not parts:
                    continue
                stack = pd.concat(parts, ignore_index=True)
                summ = stack.groupby('time_s', as_index=False)['m'].agg(mean='mean', sem='sem')
                color = CATEGORY_COLORS.get(cat)
                ax.plot(summ['time_s'], summ['mean'], color=color, lw=2, label=cat)
                ax.fill_between(summ['time_s'], summ['mean'] - summ['sem'], summ['mean'] + summ['sem'], color=color, alpha=0.18)
            ax.axhline(0.0, color='k', ls='--', lw=0.6, alpha=0.6)
            ax.set_title(f'{ctr} | {tmpl}')
            ax.set_xlabel('Cue time (s)')
            ax.set_ylabel('Reactivation (cue HFA · z-weight)')

    handles, labels = [], []
    for ax in axes.ravel():
        h, lab = ax.get_legend_handles_labels()
        if h:
            handles, labels = h, lab
            break
    if handles:
        fig.legend(handles, labels, title='Cue RPE', loc='upper right', bbox_to_anchor=(0.99, 0.98))
    fig.suptitle('All subjects | ACC RPE-category encoding reactivation', y=1.02)
    fig.tight_layout()
    out_r = parallel_figure_root / 'all_subjects_acc_rpecat_encoding_reinstatement.pdf'
    fig.savefig(out_r, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved:', out_r)

    rpecat_ps = per_subject_figure_root / 'rpecat_encoding_reactivation'
    rpecat_ps.mkdir(parents=True, exist_ok=True)
    for sid in sorted(rpecat_agg_all_subjects_df['subject'].astype(str).unique()):
        fig, axes = plt.subplots(len(contrasts), len(templates), figsize=(10, 7), sharex='col')
        for i, ctr in enumerate(contrasts):
            for j, tmpl in enumerate(templates):
                ax = axes[i, j]
                sub_all = rpecat_agg_all_subjects_df[
                    (rpecat_agg_all_subjects_df['subject'] == sid)
                    & (rpecat_agg_all_subjects_df['contrast'] == ctr)
                    & (rpecat_agg_all_subjects_df['template'] == tmpl)
                ]
                if len(sub_all) == 0:
                    ax.set_title(f'{ctr} | {tmpl}\n(no data)')
                    continue
                for cat in RPE_ORDER:
                    ssub = sub_all[sub_all['cue_rpe_cat'] == cat]
                    if len(ssub) == 0:
                        continue
                    g = (
                        ssub.groupby('time_s', as_index=False)['reactivation']
                        .agg(m='mean')
                        .sort_values('time_s')
                    )
                    color = CATEGORY_COLORS.get(cat)
                    ax.plot(g['time_s'], g['m'], color=color, lw=2, label=cat)
                ax.axhline(0.0, color='k', ls='--', lw=0.6, alpha=0.6)
                ax.set_title(f'{ctr} | {tmpl}')
                ax.set_xlabel('Cue time (s)')
                ax.set_ylabel('cue·w_z')
        handles, labels = [], []
        for ax in axes.ravel():
            h, lab = ax.get_legend_handles_labels()
            if h:
                handles, labels = h, lab
                break
        if handles:
            fig.legend(handles, labels, title='Cue RPE', loc='upper right', bbox_to_anchor=(0.99, 0.98))
        fig.suptitle(f'{sid} | ACC RPE-category encoding reactivation', y=1.02)
        fig.tight_layout()
        fig.savefig(rpecat_ps / f'{sid}_acc_rpecat_encoding_reinstatement.png', dpi=200, bbox_inches='tight')
        plt.close(fig)
    print('Saved per-subject rpecat figures to:', rpecat_ps)



In [ ]:
# -----------------------------
# Group + per-subject: HPC theta (z_beta_diff, z_mi_diff) vs ACC reactivation
# -----------------------------
def _sem_across_subjects(series):
    x = np.asarray(series, dtype=float)
    x = x[np.isfinite(x)]
    if x.size <= 1:
        return 0.0
    return float(np.nanstd(x, ddof=1) / np.sqrt(x.size))


if len(hpc_theta_all_subjects_df) == 0:
    print('No HPC theta rows; run pipeline with RUN_HPC_THETA=True or check audit hpc_theta_error.')
else:
    contrasts = ['pos_vs_neg', 'pos_vs_neutral']
    templates = ['mvpa_decoding', 'cross_decoding']
    value_cols = [('z_beta_diff', 'all_subjects_hpc_theta_zbeta_diff.pdf'), ('z_mi_diff', 'all_subjects_hpc_theta_zmi_diff.pdf')]

    for vcol, pdf_name in value_cols:
        fig, axes = plt.subplots(len(contrasts), len(templates), figsize=(10, 7), sharex='col', sharey='row')
        for i, ctr in enumerate(contrasts):
            for j, tmpl in enumerate(templates):
                ax = axes[i, j]
                sub = hpc_theta_all_subjects_df[
                    (hpc_theta_all_subjects_df['contrast'] == ctr)
                    & (hpc_theta_all_subjects_df['template'] == tmpl)
                ]
                if len(sub) == 0:
                    ax.set_title(f'{ctr} | {tmpl}\n(no data)')
                    continue
                rows = []
                for sid, g in sub.groupby('subject'):
                    gg = g.groupby('time_s', as_index=False)[vcol].mean()
                    if len(gg):
                        rows.append(gg.rename(columns={vcol: 'm'}))
                if not rows:
                    continue
                stack = pd.concat(rows, ignore_index=True)
                summ = stack.groupby('time_s', as_index=False).agg(
                    mean=('m', 'mean'),
                    sem=('m', lambda s: _sem_across_subjects(s)),
                )
                ax.plot(summ['time_s'], summ['mean'], color='C0', lw=2)
                ax.fill_between(
                    summ['time_s'],
                    summ['mean'] - summ['sem'],
                    summ['mean'] + summ['sem'],
                    color='C0',
                    alpha=0.2,
                )
                ax.axhline(0.0, color='k', ls='--', lw=0.6, alpha=0.6)
                ax.set_title(f'{ctr} | {tmpl}')
                ax.set_xlabel('Cue time (s)')
                ax.set_ylabel(vcol + '\n(mean over HPC ch) ± SEM across subjects')
        fig.suptitle(f'All subjects | HPC theta vs ACC reactivation | {vcol}', y=1.02)
        fig.tight_layout()
        outp = parallel_figure_root / pdf_name
        fig.savefig(outp, dpi=200, bbox_inches='tight')
        plt.show()
        print('Saved:', outp)

    hpc_ps = per_subject_figure_root / 'hpc_theta'
    hpc_ps.mkdir(parents=True, exist_ok=True)
    for sid in sorted(hpc_theta_all_subjects_df['subject'].astype(str).unique()):
        for vcol, stub in [('z_beta_diff', 'zbeta'), ('z_mi_diff', 'zmi')]:
            fig, axes = plt.subplots(len(contrasts), len(templates), figsize=(10, 7), sharex='col', sharey='row')
            for i, ctr in enumerate(contrasts):
                for j, tmpl in enumerate(templates):
                    ax = axes[i, j]
                    sub = hpc_theta_all_subjects_df[
                        (hpc_theta_all_subjects_df['subject'] == sid)
                        & (hpc_theta_all_subjects_df['contrast'] == ctr)
                        & (hpc_theta_all_subjects_df['template'] == tmpl)
                    ]
                    if len(sub) == 0:
                        ax.set_title(f'{ctr} | {tmpl}\n(no data)')
                        continue
                    summ = sub.groupby('time_s', as_index=False)[vcol].mean()
                    ax.plot(summ['time_s'], summ[vcol], color='C0', lw=1.5)
                    ax.axhline(0.0, color='k', ls='--', lw=0.6, alpha=0.6)
                    ax.set_title(f'{ctr} | {tmpl}')
                    ax.set_xlabel('Cue time (s)')
                    ax.set_ylabel(vcol + ' (mean HPC ch)')
            fig.suptitle(f'{sid} | HPC theta | {vcol}', y=1.02)
            fig.tight_layout()
            fig.savefig(hpc_ps / f'{sid}_hpc_theta_{stub}.png', dpi=200, bbox_inches='tight')
            plt.close(fig)
    print('Saved per-subject HPC theta figures to:', hpc_ps)



In [ ]:
combined_df_reactivation_all_subjects_df.keys()

In [ ]:
combined_df_reactivation_all_subjects_df['recog_rt_log'] = np.log(combined_df_reactivation_all_subjects_df['recog_rt'])
sns.barplot(data=combined_df_reactivation_all_subjects_df, 
x='rpe_cat', y='recog_rt_log', errorbar='se')


In [ ]:
sns.lmplot(data=model_df,
x='recog_rt_log', y='react_pos_vs_neutral_cross_cue', hue='rpe_cat', scatter=True, ci=95)

In [ ]:
model_df = combined_df_reactivation_all_subjects_df.copy()
model_df[model_df.recog_rt<1] = np.nan
model_df[model_df.recog_rt>10] = np.nan
model_df = model_df.dropna(subset=['recog_rt_log', 'react_pos_vs_neutral_cross_cue', 'rpe_cat'])


In [ ]:
import statsmodels.formula.api as smf
model = smf.mixedlm(data=model_df,
formula='recog_rt_log ~ 1 + react_pos_vs_neutral_cross_cue*C(rpe_cat,Treatment("pos"))',
groups='subject').fit()
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
model = smf.mixedlm(data=combined_df_reactivation_all_subjects_df.dropna(subset=['recog_rt_log', 'react_pos_vs_neutral_cross_cue', 'rpe_cat']),
formula='recog_rt_log ~ react_pos_vs_neg_cross_cue*C(rpe_cat,Treatment("pos"))',
groups='subject').fit()
print(model.summary())

In [ ]:
sns.barplot(data=combined_df_reactivation_all_subjects_df, 
x='rpe_cat', y='react_pos_vs_neutral_cross_cue', errorbar='se')

In [ ]:
sns.barplot(data=combined_df_reactivation_all_subjects_df, 
x='rpe_cat', y='react_pos_vs_neg_cross_cue', errorbar='se')